# Retrieval Strategy Evaluation

Compares four retrieval strategies on the competency question set (196 queries, 7 categories).

**Chunk level retrieval** runs BM25, dense, and hybrid (RRF) over the 5941 chunk Qdrant collection.
Each query has an expected book so we can check whether the top result lands in the right book.

**Book level retrieval** runs thematic search over the 9 book summaries for exploratory and
thematic queries where the goal is to identify the right book rather than a specific passage.

The output is a simple pandas table so we can eyeball scores and correctness across strategies.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import vertexai
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from rank_bm25 import BM25Okapi
from vertexai.language_models import TextEmbeddingModel

sys.path.insert(0, str(Path.cwd().parent))

from src.tools.vector_search import bm25_search, dense_search, hybrid_search
from src.tools.book_summary_search import book_summary_search
from src.models.agent_contracts import Passage

load_dotenv(Path("../.env"))
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(Path("..") / os.environ.get("google_application_credentials", ""))

vertexai.init(project=os.environ["gcp_project"], location=os.environ["gcp_location"])
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-005")
qdrant = QdrantClient(url=os.environ["qdrant_url"], api_key=os.environ["qdrant_api_key"])
collection = "literary_chunks"


## Load indexes

BM25 runs in memory over the recursive (unenriched) chunks. Dense and hybrid hit Qdrant
which already has the contextual embeddings uploaded. Book summaries are loaded separately
for thematic search.

In [ ]:
# loading the recursive chunks that bm25 indexes over
with open("../data/chunks/recursive_chunks.json") as f:
    chunks = json.load(f)

corpus_tokens = [c["text"].lower().split() for c in chunks]
bm25_index = BM25Okapi(corpus_tokens)

# load book summaries and their precomputed embeddings for thematic search
with open("../data/book_summaries.json") as f:
    summaries = json.load(f)
summary_embeddings = np.array([s["embedding"] for s in summaries], dtype=np.float32)

print(f"bm25 index: {len(chunks)} chunks")
print(f"book summaries: {len(summaries)} books")
print(f"summary embedding shape: {summary_embeddings.shape}")

## Load competency questions

196 questions across 7 categories. Each has an expected book so we know the ground truth
scope. We sample 3 per category to keep the evaluation quick for a demo run, but you can
set `sample_n = None` to run the full set.

In [ ]:
with open("../data/competency_questions.json") as f:
    all_cqs = json.load(f)

# group by category so we can sample evenly
cqs_by_cat = {}
for cq in all_cqs:
    cqs_by_cat.setdefault(cq["category"], []).append(cq)

# set to None to use every question instead of a sample
sample_n = 3

if sample_n:
    import random
    random.seed(42)
    eval_cqs = []
    for cat, items in sorted(cqs_by_cat.items()):
        eval_cqs.extend(random.sample(items, min(sample_n, len(items))))
else:
    eval_cqs = all_cqs

print(f"evaluation set: {len(eval_cqs)} questions from {len(cqs_by_cat)} categories")
for cat, items in sorted(cqs_by_cat.items()):
    n = sum(1 for cq in eval_cqs if cq["category"] == cat)
    print(f"  {cat}: {n}")

## Evaluation helpers

We measure two things per query per method:
1. **book_hit** whether the top 1 result comes from the expected book (proxy for relevance)
2. **top_score** the retrieval confidence of the top result

These are simple but informative. Book hit tells us if the method even looked in the right
place. Top score tells us how confident it was. Together they separate lucky guesses from
strong retrievals.

In [ ]:
def run_bm25(query, book_id=None, top_k=5):
    """
    run bm25 keyword search over the in memory index.
    """
    return bm25_search(query, bm25_index, chunks, book_id=book_id, top_k=top_k)


def run_dense(query, book_id=None, top_k=5):
    """
    run dense cosine similarity search via qdrant.
    """
    return dense_search(query, qdrant, collection, embedding_model, book_id=book_id, top_k=top_k)


def run_hybrid(query, book_id=None, top_k=5):
    """
    run hybrid search fusing bm25 and dense via reciprocal rank fusion.
    """
    return hybrid_search(query, bm25_index, chunks, qdrant, collection, embedding_model, book_id=book_id, top_k=top_k)


def run_thematic(query, top_k=3):
    """
    run dense search over book level summaries for broad thematic queries.
    """
    return book_summary_search(query, summaries, summary_embeddings, embedding_model, top_k=top_k)


def check_book_hit(passages, expected_books):
    """
    check if the top passage comes from one of the expected books.
    we only look at rank 1 because that is what the user actually sees first.
    """
    if not passages:
        return False
    return passages[0].book_id in expected_books


def extract_top_score(passages):
    """
    pull the score from the top ranked passage or return 0 if empty.
    """
    return passages[0].score if passages else 0.0


def snippet(passages, max_len=80):
    """
    grab a short text preview from the top passage for the results table.
    """
    if not passages:
        return ""
    return passages[0].text[:max_len].replace("\n", " ")


print("helpers defined")

## Run chunk level evaluation (BM25 / Dense / Hybrid)

Each query runs through all three methods without any book filter. This is the hardest
setting because the method has to find the right book out of 5941 chunks across 9 books
with no hints. We record the top 1 book hit, score, and a text snippet per method.

In [ ]:
# we collect one row per query with results from all three chunk level methods
# storing passages separately lets us inspect them later without rerunning search
rows = []
methods = {"bm25": run_bm25, "dense": run_dense, "hybrid": run_hybrid}

for i, cq in enumerate(eval_cqs):
    q = cq["question"]
    expected = cq["example_books"]
    row = {"id": cq["id"], "category": cq["category"], "question": q[:90], "expected_book": expected[0]}

    for name, fn in methods.items():
        t0 = time.time()
        passages = fn(q, top_k=5)
        elapsed_ms = (time.time() - t0) * 1000
        row[f"{name}_hit"] = check_book_hit(passages, expected)
        row[f"{name}_score"] = round(extract_top_score(passages), 4)
        row[f"{name}_book"] = passages[0].book_id if passages else ""
        row[f"{name}_ms"] = round(elapsed_ms)

    rows.append(row)
    if (i + 1) % 5 == 0:
        print(f"  processed {i + 1}/{len(eval_cqs)}")

chunk_df = pd.DataFrame(rows)
print(f"\ndone: {len(chunk_df)} queries evaluated")

## Chunk level results table

Full per query breakdown showing which method hit the right book and its confidence score.

In [ ]:
# selecting only the columns that matter for a quick visual scan
display_cols = ["id", "category", "question", "expected_book", "bm25_hit", "bm25_score", "dense_hit", "dense_score", "hybrid_hit", "hybrid_score"]
chunk_df[display_cols].style.apply(lambda col: ["background-color: #d4edda" if v else "background-color: #f8d7da" if v is False else "" for v in col],
                                    subset=["bm25_hit", "dense_hit", "hybrid_hit"])

## Chunk level aggregate metrics

Book accuracy at rank 1 (what fraction of queries had the top result from the correct book),
mean top score, and mean latency. Grouped by method and also broken down by category to see
where each strategy struggles.

In [ ]:
def aggregate_metrics(df, method):
    """
    computing book accuracy, mean score, and mean latency for one method across the dataframe.
    """
    return {"method": method, "book_accuracy": df[f"{method}_hit"].mean(), "mean_score": df[f"{method}_score"].mean(), "mean_latency_ms": df[f"{method}_ms"].mean()}

# overall metrics across all query categories
overall = pd.DataFrame([aggregate_metrics(chunk_df, m) for m in ["bm25", "dense", "hybrid"]])
print("overall chunk level metrics:")
print(overall.to_string(index=False))
print()


rows_by_cat = []

for cat in sorted(chunk_df["category"].unique()):
    cat_df = chunk_df[chunk_df["category"] == cat]
    
    for method in ["bm25", "dense", "hybrid"]:
        r = aggregate_metrics(cat_df, method)
        r["category"] = cat
        r["n"] = len(cat_df)
        rows_by_cat.append(r)

cat_metrics = pd.DataFrame(rows_by_cat)[["category", "n", "method", "book_accuracy", "mean_score", "mean_latency_ms"]]
cat_metrics.style.format({"book_accuracy": "{:.0%}", "mean_score": "{:.4f}", "mean_latency_ms": "{:.0f}"})

## Thematic (book level) evaluation

Thematic search operates at a different granularity. Instead of returning chunks it returns
whole book summaries ranked by cosine similarity. This is the right tool for exploratory
queries like "which books explore the theme of ambition" where the answer is a book not a
passage. We evaluate on thematic and comparative CQs since those are the categories the
routing table would send to the thematic agent.

In [ ]:
# filter to thematic and comparative queries since those are the ones
# that would actually be routed to the thematic agent in production
thematic_cats = {"Thematic", "Comparative"}
thematic_cqs = [cq for cq in eval_cqs if cq["category"] in thematic_cats]

thematic_rows = []
for cq in thematic_cqs:
    q = cq["question"]
    expected = cq["example_books"]
    t0 = time.time()
    passages = run_thematic(q, top_k=3)
    elapsed_ms = (time.time() - t0) * 1000

    # for comparative queries the expected_books has multiple entries
    top_books = [p.book_id for p in passages]
    hit = any(b in top_books for b in expected)
    all_hit = all(b in top_books for b in expected)

    thematic_rows.append({"id": cq["id"], "category": cq["category"], "question": q[:90],
                            "expected_books": ", ".join(expected), "returned_books": ", ".join(top_books),
                            "any_hit": hit, "all_hit": all_hit, "top_score": round(passages[0].score, 4) if passages else 0.0,
                            "latency_ms": round(elapsed_ms)})

thematic_df = pd.DataFrame(thematic_rows)
print(f"thematic evaluation: {len(thematic_df)} queries")

thematic_df.style.apply(lambda col: ["background-color: #d4edda" if v else "background-color: #f8d7da" if v is False else "" for v in col], subset=["any_hit", "all_hit"])

## Head to head: same query across all methods

Pick a few representative queries and show the top result from every strategy side by side.
This is the most intuitive view for a demo because you can read the actual text each method
retrieved and judge quality yourself.

In [ ]:
head_to_head_queries = [
    {"label": "factual (entity heavy)", "query": "Who is the narrator of The Great Gatsby?", "expected": "great_gatsby"},
    {"label": "fuzzy (no entity names)", "query": "a scientist who creates something monstrous and then suffers for it", "expected": "frankenstein"},
    {"label": "relational", "query": "What is the relationship between Elizabeth Bennet and Mr Darcy?", "expected": "pride_and_prejudice"},
    {"label": "temporal", "query": "What happens after Dracula arrives in England on the ship Demeter?", "expected": "dracula"},
    {"label": "mixed (entity + semantic)", "query": "Beowulf fights Grendel in the great hall", "expected": "beowulf"}]

h2h_rows = []
for item in head_to_head_queries:
    q = item["query"]
    for name, fn in methods.items():
        passages = fn(q, top_k=1)
        h2h_rows.append({
            "label": item["label"],
            "query": q[:70],
            "method": name,
            "book_hit": passages[0].book_id == item["expected"] if passages else False,
            "top_book": passages[0].book_id if passages else "",
            "score": round(extract_top_score(passages), 4),
            "snippet": snippet(passages, max_len=100)})

    passages = run_thematic(q, top_k=1)
    h2h_rows.append({
        "label": item["label"],
        "query": q[:70],
        "method": "thematic",
        "book_hit": passages[0].book_id == item["expected"] if passages else False,
        "top_book": passages[0].book_id if passages else "",
        "score": round(extract_top_score(passages), 4),
        "snippet": snippet(passages, max_len=100)})

h2h_df = pd.DataFrame(h2h_rows)
h2h_df.style.apply(lambda col: ["background-color: #d4edda" if v else "background-color: #f8d7da" if v is False else "" for v in col], subset=["book_hit"])

## Summary

Final comparison table with one row per method showing overall book accuracy at rank 1,
mean retrieval score, and mean latency. This is the table you show in a deck.

In [ ]:
summary_rows = []

for method in ["bm25", "dense", "hybrid"]:
    summary_rows.append({"method": method, "book_accuracy_at_1": f"{chunk_df[f'{method}_hit'].mean():.0%}",
                        "mean_score": round(chunk_df[f"{method}_score"].mean(), 4), "mean_latency_ms": round(chunk_df[f"{method}_ms"].mean()), "scope": "chunk"})

# adding thematic summary if we ran it
if len(thematic_df) > 0:
    summary_rows.append({"method": "thematic", "book_accuracy_at_1": f"{thematic_df['any_hit'].mean():.0%}",
                        "mean_score": round(thematic_df["top_score"].mean(), 4), "mean_latency_ms": round(thematic_df["latency_ms"].mean()),"scope": "book"})

summary_df = pd.DataFrame(summary_rows)
print("retrieval strategy comparison")
print("=" * 70)
print(summary_df.to_string(index=False))
print()
print("book_accuracy_at_1 = fraction of queries where rank 1 result was from the expected book")
print("scope = granularity of retrieval (chunk = 512 token passage, book = full summary)")